In [1]:
import numpy as np
import pandas as pd
import sys
from matplotlib import pyplot as plt
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
sys.path.insert(0, '../raw_data_elaboration/tools/')

import data_organisation as do


In [2]:
%config InteractiveShell.ast_node_interactivity = 'all'

In [3]:
class dataset(object):
    def __init__(self, data_filepath, metadata_filepath):
        
        self.df_temp = pd.read_pickle(data_filepath)
        self.meta_temp = pd.read_pickle(metadata_filepath)
        
        self.df = dict()
        self.meta_dict = dict()
        self.species = dict()
        self.sites = dict()
        self.ts_properties = dict()
        self.series_length_in_years = []

        index_list = []
        for e in self.df_temp:
            if e.shape[0] > 0:
                series_id = e.series_id.iloc[0]
                meta_row = self.meta_temp[self.meta_temp.series_id == series_id]
                index_list.append(meta_row.index[0])
                length = e.shape[0]
                len_days = length / 144.0
                len_years = length / 52560.0
                series_start = e.ts.iloc[0]
                series_end = e.ts.iloc[-1]
                species = meta_row.tree_species
                site = meta_row.site_name
                self.ts_properties[series_id] = [length, len_days, len_years, series_start, series_end, species, site]
                self.df[series_id] = e
        
        self.meta_df = self.meta_temp.loc[index_list] # note: it is important to use 'loc' and not 'iloc'

        genus_list = self.meta_df.tree_genus.drop_duplicates().to_list()
        for e in genus_list:
            temp_df = self.meta_df[self.meta_df.tree_genus == e]
            self.species[e] = [temp_df.tree_species.drop_duplicates().to_list(), temp_df.shape[0]]

        site_list = self.meta_df.site_name.drop_duplicates().to_list()
        for e in site_list:
            temp_df = self.meta_df[self.meta_df.site_name == e]
            species = temp_df.tree_species.drop_duplicates().to_list()
            ids = temp_df.series_id.to_list()
            species_count = []
            for ee in species:
                species_count.append(temp_df[temp_df.tree_species == ee].shape[0])
            siteXcor = pd.to_numeric(temp_df.iloc[0].site_xcor)
            siteYcor = pd.to_numeric(temp_df.iloc[0].site_ycor)
            self.sites[e] = [temp_df.shape[0], species, species_count, [siteXcor, siteYcor], ids]


    def get_metadata_parameters(self):
        parameters = list(self.meta_df)
        
        return parameters
    
    def get_ts_length_distro(self):
        for e in self.ts_properties.values():
            self.series_length_in_years.append(e[2])
        
        return self.series_length_in_years

    def get_site_ts(self, name):
        series_id_list = self.sites[name][4]
        ts_collection = []
        for e in series_id_list:
            ts_collection.append(self.df[e])

        return ts_collection

    def get_df(self):
        return self.df

    def get_sites(self):
        return self.sites

    def get_meta_dict(self):
        for e in self.meta_df:
            self.meta_dict[e.index_id] = e
        return self.meta_dict

    def get_meta_df(self):
        return self.meta_df

    def get_meta_original(self):
        return self.meta_temp


In [4]:
def save_csv(dictionary, site):
    for e in dictionary:
        tree_id = e.series_id[0]
        file_name = 'time_series_' + site + '_' + str(tree_id) + '.csv'
        e.to_csv(file_name)

In [131]:
metadata_path = "/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata.pkl"
data_path = "/storage/lukovic/Data/FORWARDS/treenet/server_data/data_dendro_lm.pkl"

#df = dataset(data_path, metadata_path)

In [6]:
data = df.get_df()
meta = df.get_meta_df()
meta_old = df.get_meta_original()

In [19]:
list(meta)

['measure_point',
 'series_id',
 'series_start',
 'series_stop',
 'series_cutout',
 'series_active',
 'series_display',
 'variable_id',
 'variable_name',
 'variable_resolution',
 'variable_units',
 'sensor_id',
 'sensor_name',
 'sensor_class',
 'sensor_data_source',
 'position_id',
 'series_height',
 'series_exposition',
 'series_distance_to_tree',
 'tree_id',
 'tree_name',
 'tree_xcor',
 'tree_ycor',
 'tree_altitude',
 'tree_genus',
 'tree_species',
 'tree_dbh',
 'tree_height',
 'tree_status',
 'tree_age',
 'tree_phloem_thickness_mm',
 'tree_totalbark_thickness_mm',
 'tree_sapwood_thickness_cm',
 'tree_sapwood_area',
 'series_dsri_max',
 'series_dsri_min',
 'series_twd_max_gp',
 'series_twd_med_gp',
 'series_twd_min_gp',
 'series_twd_max_nogp',
 'series_twd_med_nogp',
 'series_twd_min_nogp',
 'series_twd_max_frost',
 'series_twd_med_frost',
 'series_twd_min_frost',
 'series_gro_start_doy_med',
 'series_gro_end_doy_med',
 'series_gro_max_yr',
 'series_gro_med_yr',
 'series_gro_min_yr',

In [20]:
meta_old

,measure_point,series_id,series_start,series_stop,series_cutout,series_active,series_display,variable_id,variable_name,variable_resolution,...,site_annual_rad,site_growth_rad,site_annual_relh,site_growth_relh,site_annual_vpd,site_growth_vpd,site_n_depo,site_ozon,site_nfk,site_temp_ref
0,alvaneu-high_dendrometer_µm_n3_0_1.5_south_psy...,1,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,106.00,1178.0
1,alvaneu-high_dendrometer_µm_n4_0_1.5_south_psy...,2,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,106.00,1178.0
2,alvaneu-low_dendrometer_µm_n1_0_1.5_south_psyl...,3,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,191.00,1178.0
3,alvaneu-low_dendrometer_µm_n2_0_1.5_south_psyl...,4,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,191.00,1178.0
4,bachtel-forest_precipitation_mm_meteo_-999_2_b...,5,2012-08-24,2022-09-28,None,no,no,10,precipitation,10,...,None,None,None,None,None,None,None,None,None,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1820,zürich-rooftop_dendrometer_µm_föhre-dach-ast2_...,1825,2015-04-02,2015-12-15,None,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,None,1447.0
1821,choeliacher-orchard_soil_temp_°c_11_-9995_-0.1...,1826,2018-12-06,2024-05-07,None,no,yes,29,soil temperature,10,...,None,None,None,None,None,None,None,None,None,119.0
1822,choeliacher-orchard_soil_wp_kpa_11_-9995_-0.1_...,1827,2018-12-06,2024-05-07,None,no,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,119.0
1823,jussy-forest_dendrometer_µm_baum-97_0_1.5_nort...,1828,2021-05-28,2024-05-28,01.08.2021 00:00 - 19.05.2022 09:00,no,no,3,tree stem radius change,10,...,None,None,None,None,None,None,None,None,None,571.0


In [150]:
davos_ts = df.get_site_ts('Davos-Seehornwald')

In [151]:
len(davos_ts)

45

In [160]:
save_csv(davos_ts, 'Davos-Seehornwald')

In [154]:
davos_ts[0]

,series_id,ts,value
0,221,1998-02-11 18:10:00,-985.881500
1,221,1998-02-11 18:20:00,-987.317333
2,221,1998-02-11 18:30:00,-987.519833
3,221,1998-02-11 18:40:00,-990.357000
4,221,1998-02-11 18:50:00,-993.194167
...,...,...,...
484605,221,2007-06-04 23:10:00,10908.359670
484606,221,2007-06-04 23:20:00,10908.301170
484607,221,2007-06-04 23:30:00,10904.472170
484608,221,2007-06-04 23:40:00,10911.782170


In [162]:
meta_davos = meta[meta.site_name == 'Davos-Seehornwald']

In [165]:
meta_davos.to_csv('metadata_Davos-Seehornwald.csv')

# Data organisation

In [2]:
import pandas as pd
import pickle


def load_data(meta_path, df_path):
    metatemp = pd.read_pickle(meta_path)
    datatemp = pd.read_pickle(df_path)

    metadata = dict()
    dataframe = dict()

    for e in datatemp:
        if len(e) > 0:
            key = e.series_id.iloc[0]
            dataframe[key] = e
            metadata[key] = metatemp[metatemp.series_id == key]

    return metadata, dataframe


def get_species(metadata):
    species = dict()
    genus_list = metadata.tree_genus.drop_duplicates().to_list()
    for e in genus_list:
        temp_df = metadata[metadata.tree_genus == e]
        species[e] = [temp_df.tree_species.drop_duplicates().to_list(), len(temp_df)]
        # Todo: Include the number of species trees

    return species


def get_sites(metadata):
    sites = dict()
    site_list = metadata.site_name.drop_duplicates().to_list()
    for e in site_list:
        temp_df = metadata[metadata.site_name == e]
        species = temp_df.tree_species.drop_duplicates().to_list()
        species_count = []
        for ee in species:
            species_count.append(temp_df[temp_df.tree_species == ee].shape[0])
        siteXcor = pd.to_numeric(temp_df.iloc[0].site_xcor)
        siteYcor = pd.to_numeric(temp_df.iloc[0].site_ycor)
        sites[e] = [species, temp_df.shape[0], species_count, [siteXcor, siteYcor]]

    return sites


def get_yearly_data_by_id(dictionary):
    # NOTE: Separates the time series of each sensor into a list of time series by year. Creates a dictionary where the key is the series id and the value is 
    #  a list of time series by year.
    # NOTE: take care of leap years, they are also included
    data_yr = dict()
    for key, value in dictionary.items():
        df_list = _ts_by_year(value)
        temp_list = []
        for e in df_list[:]:
            if e.doy.iloc[0] == 1 or e.doy.iloc[-1] > 364:
                temp_list.append(e)
        data_yr[key] = temp_list
    return data_yr


def get_yearly_data_by_year(dictionary, yearly_by_id):
    # NOTE: Sorts the yearly time series according to year. Creates a dictionary where the key is the year and the value are time series of different sensors 
    #  of the same year. 
    # NOTE: take care of leap years, they are also included
    if not yearly_by_id:
        dictionary = get_yearly_data_by_id(dictionary)
    data_by_year = dict()
    for key, value in dictionary.items():
        for e in value:
            year = e.year.iloc[0]
            if year in data_by_year:
                data_by_year[year].append(e)
            else:
                data_by_year[year] = [e]
    return data_by_year


def _ts_by_year(data):
    data.loc[:, 'year'] = data.ts.dt.year  # note: add a column with the year
    data.loc[:, 'doy'] = data.ts.dt.dayofyear  # note: add a column with the day of year
    data.loc[:, 'hour'] = data.ts.dt.hour  # note: add a column with the hour of day
    years = data.year.unique()
    output = []
    for year in years:
        output.append(data[data.year == year])

    return output

In [3]:
df = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_dendro_lm.pkl")
meta = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_dendrometer.pkl")
meta_all = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")

In [ ]:
df[4]

,series_id,ts,value
0,4,2014-05-21 12:40:00,3772.214407
1,4,2014-05-21 12:50:00,3768.554000
2,4,2014-05-21 13:00:00,3765.468070
3,4,2014-05-21 13:10:00,3762.413673
4,4,2014-05-21 13:20:00,3759.363333
...,...,...,...
435216,4,2022-10-05 00:10:00,11582.641602
435217,4,2022-10-05 00:20:00,11582.641602
435218,4,2022-10-05 00:30:00,11582.641602
435219,4,2022-10-05 00:40:00,11582.641602


In [43]:
data_y = get_yearly_data_by_id(df)
data_yr = get_yearly_data_by_year(data_y, True)

In [19]:
data_y[1][1]

,series_id,site_id,ts,stem_radius,temp,rh,swp,total_precip,rad,vpd,year,doy,hour
32324,1.0,1.0,2015-01-01 00:00:00,NaN,-8.500000,88.833330,-10.348725,0.000000,0.000000,0.036122,2015,1,0
32325,NaN,NaN,2015-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,1,0
32326,NaN,NaN,2015-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,1,0
32327,NaN,NaN,2015-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,1,0
32328,NaN,NaN,2015-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84879,NaN,NaN,2015-12-31 23:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,365,23
84880,NaN,NaN,2015-12-31 23:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,365,23
84881,NaN,NaN,2015-12-31 23:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,365,23
84882,NaN,NaN,2015-12-31 23:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,365,23


In [15]:
data_yr[2002][3][0:6]

,series_id,site_id,ts,stem_radius,temp,rh,swp,total_precip,rad,vpd,year,doy,hour
233561,277.0,277.0,2002-01-01 00:00:00,3835.835000,-15.300000,73.600000,None,None,0.000000,0.049432,2002,1,0
233562,277.0,277.0,2002-01-01 00:10:00,3835.670000,NaN,NaN,NaN,NaN,NaN,NaN,2002,1,0
233563,277.0,277.0,2002-01-01 00:20:00,3835.655000,NaN,NaN,NaN,NaN,NaN,NaN,2002,1,0
233564,277.0,277.0,2002-01-01 00:30:00,3835.445000,NaN,NaN,NaN,NaN,NaN,NaN,2002,1,0
233565,277.0,277.0,2002-01-01 00:40:00,3835.280000,NaN,NaN,NaN,NaN,NaN,NaN,2002,1,0
233566,277.0,277.0,2002-01-01 00:50:00,3835.175000,NaN,NaN,NaN,NaN,NaN,NaN,2002,1,0


In [26]:
element = data_yr[2002][0]

new = element.groupby(['doy', 'hour'])

list = []
for _, b in new:
    list.append(b)

df = {'index': [], 'series_id': [], 'ts': [], 'year': [], 'doy': [], 'hour': [], 'stem_radius': []}

for e in list:
    df['index'].append(e.index[0])
    df['series_id'].append(e.series_id.iloc[0])
    df['ts'].append(e.ts.iloc[0])
    df['year'].append(e.year.iloc[0])
    df['doy'].append(e.doy.iloc[0])
    df['hour'].append(e.hour.iloc[0])
    df['stem_radius'].append(e.stem_radius.mean())

df_hourly = pd.DataFrame(df)
df_hourly.set_index('index', inplace=True)
df_hourly.index.name = ""
df_hourly

,series_id,ts,year,doy,hour,stem_radius
,,,,,,
204371,221.0,2002-01-01 00:00:00,2002,1,0,4007.506167
204377,221.0,2002-01-01 01:00:00,2002,1,1,4003.127834
204383,221.0,2002-01-01 02:00:00,2002,1,2,3998.522834
204389,221.0,2002-01-01 03:00:00,2002,1,3,3993.111167
204395,221.0,2002-01-01 04:00:00,2002,1,4,3987.445334
...,...,...,...,...,...,...
256901,221.0,2002-12-31 19:00:00,2002,365,19,6024.210167
256907,221.0,2002-12-31 20:00:00,2002,365,20,6024.207667
256913,221.0,2002-12-31 21:00:00,2002,365,21,6024.198167


In [2]:
def _average_over_hour(df_in):
    d_out = {'index': [], 'series_id': [], 'ts': [], 'year': [], 'doy': [], 'hour': [], 'stem_radius': []}
    for _, e in df_in:
        d_out['index'].append(e.index[0])
        d_out['series_id'].append(e.series_id.iloc[0])
        d_out['ts'].append(e.ts.iloc[0])
        d_out['year'].append(e.year.iloc[0])
        d_out['doy'].append(e.doy.iloc[0])
        d_out['hour'].append(e.hour.iloc[0])
        d_out['stem_radius'].append(e.stem_radius.mean())
    df_hourly = pd.DataFrame(d_out)
    df_hourly.set_index('index', inplace=True)
    df_hourly.index.name = ""
    return df_hourly

def _average_over_day(df_in):
    d_out = {'index': [], 'series_id': [], 'ts': [], 'year': [], 'doy': [], 'stem_radius': []}
    for _, e in df_in:
        d_out['index'].append(e.index[0])
        d_out['series_id'].append(e.series_id.iloc[0])
        d_out['ts'].append(e.ts.iloc[0])
        d_out['year'].append(e.year.iloc[0])
        d_out['doy'].append(e.doy.iloc[0])
        d_out['stem_radius'].append(e.stem_radius.mean())
    df_daily = pd.DataFrame(d_out)
    df_daily.set_index('index', inplace=True)
    df_daily.index.name = ""
    return df_daily        

def get_hourly_data_by_id_index(dictionary):
    """ The function averages the data over every hour.
    Input: dictionary where the key is the signal id and value is a time series with 10 min resolution
    Output: dictionary where the key is the signal id and the value is a time series with 1 h resolution
    IMPORTANT: make sure that the dataframes have DOY and hour columns
    """
    
    new_dict = dict()
    for id, timeseries in dictionary.items():  # note: Iterates over the years
        temp = timeseries.groupby(['doy', 'hour'])
        new_dict[id] = _average_over_hour(temp)  # note: appends 
    return new_dict


def get_hourly_data_by_year_index(dictionary):
    """ The function averages the data over every hour.
    Input: dictionary where the key is the year and value is a list of yearly data with 10 min resolution
    Output: dictionary where the key is the year and the value is a list of yearly data with 1 h resolution
    IMPORTANT: make sure that the dataframes have DOY and hour columns
    """
    new_dict = dict()
    for year, list in dictionary.items():  # note: Iterates over the years
        new_dict[year] = []
        for e in list:  # note: Iterates over different trees for the same year
            temp = e.groupby(['doy', 'hour'])
            new_dict[year].append(_average_over_hour(temp))  # note: appends 
    return new_dict


def get_daily_data_by_id_index(dictionary, hourly_data):
    """ The function averages the data over every day.
    Input: dictionary where the key is the sensor id and value is a list of yearly data with 1h resolution
    Output: dictionary where the key is the sensor id and the value is a list of yearly data with 1 day resolution
    """
    new_dict = dict()
    if not hourly_data:
        dictionary = get_hourly_data_by_id_index(dictionary)
    for id, timeseries in dictionary.items():
        temp = timeseries.groupby(['doy'])
        new_dict[id] = (_average_over_day(temp))
    return (new_dict)


def get_daily_data_by_year_index(dictionary, hourly_data):
    """ The function averages the data over every day.
    Input: dictionary where the key is the year and value is a list of yearly data with 1h resolution
    Output: dictionary where the key is the year and the value is a list of yearly data with 1 day resolution
    """
    new_dict = dict()
    if not hourly_data:
        dictionary = get_hourly_data_by_year_index(dictionary)
    for year, list in dictionary.items():
        new_dict[year] = []
        for e in list:
            temp = e.groupby(['doy'])
            new_dict[year].append(_average_over_day(temp))
    return (new_dict)



In [3]:
import pickle

element = data_yr[2002][0]
new = element.groupby(['doy', 'hour'])
df_hourly_new = _average_over_hour(new)
new = element.groupby(['doy'])
df_daily_new = _average_over_day(new)

with open("df_list_by_year_hourly.pkl", "wb") as fp:   #Pickling
    pickle.dump(df_hourly_new, fp)

with open("df_list_by_year_daily.pkl", "wb") as fp:   #Pickling
    pickle.dump(df_daily_new, fp)


NameError: name 'data_yr' is not defined

In [18]:
data = df[35]
data.loc[:, 'year'] = data.ts.dt.year  # note: add a column with the year
data.loc[:, 'doy'] = data.ts.dt.dayofyear  # note: add a column with the day of year
data.loc[:, 'hour'] = data.ts.dt.hour  # note: add a column with the hour of day

entry = dict()
entry[1] = data

#hourly = get_hourly_data_by_id_index(entry)

AttributeError: Can only use .dt accessor with datetimelike values

In [35]:
for index, value in df.items():
    if len(value) == 0:
        print(index)

35
42
45
48
86
87
88
89
90
95
98
101
104
109
112
115
140
147
152
153
360
361
368
371
374
411
412
562
563
569
700
701
702
703
704
705
706
707
718
719
720
721
722
723
724
725
738
739
740
741
742
743
744
745
756
757
758
759
760
761
762
763
782
783
784
785
786
787
788
789
800
801
802
803
822
823
824
825
826
883
888
891
894
897
902
905
908
1246
1247
1248
1249
1250
1251
1260
1261
1262
1263
1264
1265
1302
1303
1304
1317
1318
1319
1328
1329
1330
1331
1340
1341
1342
1409
1414
1417
1420
1423
1430
1433
1436
1439
1455
1456
1457
1470
1471
1472
1479
1480
1481
1490
1491
1492
1501
1502
1503
1516
1517
1518
1527
1528
1529
1536
1537
1538
1547
1548
1549
1550
1551
1552
1553
1554
1555
1556
1557
1558
1559
1560
1561
1562
1563
1564
1565
1566
1567
1568
1569
1570
1571
1572
1573
1574
1575
1576
1577
1578
1579
1580
1581
1582
1583
1584
1585
1586
1587
1588
1589
1590
1591
1600
1601
1602
1603
1604
1605
1606
1607
1608
1609
1610
1611
1612
1613
1614
1615
1616
1617
1620
1621
1630
1631
1632
1633
1634
1635
1636
1637
1638
163

In [32]:
df[1]

,series_id,ts,value,year,doy,hour
0,1,2014-05-21 12:40:00,2249.375700,2014,141,12
1,1,2014-05-21 12:50:00,2248.375100,2014,141,12
2,1,2014-05-21 13:00:00,2245.998500,2014,141,13
3,1,2014-05-21 13:10:00,2243.332800,2014,141,13
4,1,2014-05-21 13:20:00,2242.045700,2014,141,13
...,...,...,...,...,...,...
438930,1,2022-10-05 00:10:00,4969.418799,2022,278,0
438931,1,2022-10-05 00:20:00,4969.418799,2022,278,0
438932,1,2022-10-05 00:30:00,4969.418799,2022,278,0
438933,1,2022-10-05 00:40:00,4969.418799,2022,278,0


In [17]:
do.get_hourly_data_by_id_index(entry)

{1:         series_id                  ts  year  doy  hour        value
                                                                    
 2               1 2014-05-21 13:00:00  2014  141    13  2242.282262
 8               1 2014-05-21 14:00:00  2014  141    14  2233.775940
 14              1 2014-05-21 15:00:00  2014  141    15  2228.491836
 20              1 2014-05-21 16:00:00  2014  141    16  2230.199130
 26              1 2014-05-21 17:00:00  2014  141    17  2238.437131
 ...           ...                 ...   ...  ...   ...          ...
 438905          1 2022-10-04 20:00:00  2022  277    20  4966.189183
 438911          1 2022-10-04 21:00:00  2022  277    21  4968.803700
 438917          1 2022-10-04 22:00:00  2022  277    22  4969.622080
 438923          1 2022-10-04 23:00:00  2022  277    23  4969.418799
 438929          1 2022-10-05 00:00:00  2022  278     0  4969.418799
 
 [73160 rows x 6 columns]}

In [9]:
data

,series_id,ts,value,year,doy,hour
0,1,2014-05-21 12:40:00,2249.375700,2014,141,12
1,1,2014-05-21 12:50:00,2248.375100,2014,141,12
2,1,2014-05-21 13:00:00,2245.998500,2014,141,13
3,1,2014-05-21 13:10:00,2243.332800,2014,141,13
4,1,2014-05-21 13:20:00,2242.045700,2014,141,13
...,...,...,...,...,...,...
438930,1,2022-10-05 00:10:00,4969.418799,2022,278,0
438931,1,2022-10-05 00:20:00,4969.418799,2022,278,0
438932,1,2022-10-05 00:30:00,4969.418799,2022,278,0
438933,1,2022-10-05 00:40:00,4969.418799,2022,278,0


## Convert pandas dataframe

In [5]:
for key, value in data.items():
    for e in value:
        e.to_csv('Y'+str(key)+'_ID'+str(e.series_id.iloc[0])+'.csv', index=False)

## Test Area

In [69]:
element.groupby(['doy', 'hour']).first()

series_id                  ts        value  year
doy hour                                                  
1   0           221 2002-01-01 00:00:00  4013.633667  2002
    1           221 2002-01-01 01:00:00  4005.033667  2002
    2           221 2002-01-01 02:00:00  4000.088667  2002
    3           221 2002-01-01 03:00:00  3995.533667  2002
    4           221 2002-01-01 04:00:00  3990.753667  2002
...             ...                 ...          ...   ...
365 19          221 2002-12-31 19:00:00  6024.260167  2002
    20          221 2002-12-31 20:00:00  6024.305167  2002
    21          221 2002-12-31 21:00:00  6024.156667  2002
    22          221 2002-12-31 22:00:00  6024.176167  2002
    23          221 2002-12-31 23:00:00  6023.868667  2002

[8760 rows x 4 columns]

In [71]:
element.groupby(['doy']).first()

,series_id,ts,value,year,hour
doy,,,,,
1,221,2002-01-01,4013.633667,2002,0
2,221,2002-01-02,3966.493667,2002,0
3,221,2002-01-03,3984.418667,2002,0
4,221,2002-01-04,3974.173667,2002,0
5,221,2002-01-05,3955.968667,2002,0
...,...,...,...,...,...
361,221,2002-12-27,6112.194667,2002,0
362,221,2002-12-28,5997.557167,2002,0
363,221,2002-12-29,5969.811667,2002,0


In [68]:
element

,series_id,ts,value,year,doy,hour
199346,221,2002-01-01 00:00:00,4013.633667,2002,1,0
199347,221,2002-01-01 00:10:00,4009.783667,2002,1,0
199348,221,2002-01-01 00:20:00,4005.933667,2002,1,0
199349,221,2002-01-01 00:30:00,4005.408667,2002,1,0
199350,221,2002-01-01 00:40:00,4005.228667,2002,1,0
...,...,...,...,...,...,...
251901,221,2002-12-31 23:10:00,6023.801167,2002,365,23
251902,221,2002-12-31 23:20:00,6023.829667,2002,365,23
251903,221,2002-12-31 23:30:00,6023.774167,2002,365,23
251904,221,2002-12-31 23:40:00,6023.810167,2002,365,23


In [72]:
element = data_yr[2002][0]
element
new = element.groupby(['doy'])
list = []
indx = []
for a, b in new:
    list.append(b)
    indx.append(a)

,series_id,ts,value,year,doy,hour
199346,221,2002-01-01 00:00:00,4013.633667,2002,1,0
199347,221,2002-01-01 00:10:00,4009.783667,2002,1,0
199348,221,2002-01-01 00:20:00,4005.933667,2002,1,0
199349,221,2002-01-01 00:30:00,4005.408667,2002,1,0
199350,221,2002-01-01 00:40:00,4005.228667,2002,1,0
...,...,...,...,...,...,...
251901,221,2002-12-31 23:10:00,6023.801167,2002,365,23
251902,221,2002-12-31 23:20:00,6023.829667,2002,365,23
251903,221,2002-12-31 23:30:00,6023.774167,2002,365,23
251904,221,2002-12-31 23:40:00,6023.810167,2002,365,23


In [49]:
list

[        series_id                  ts        value  year  doy  hour
 199346        221 2002-01-01 00:00:00  4013.633667  2002    1     0
 199347        221 2002-01-01 00:10:00  4009.783667  2002    1     0
 199348        221 2002-01-01 00:20:00  4005.933667  2002    1     0
 199349        221 2002-01-01 00:30:00  4005.408667  2002    1     0
 199350        221 2002-01-01 00:40:00  4005.228667  2002    1     0
 199351        221 2002-01-01 00:50:00  4005.048667  2002    1     0,
         series_id                  ts        value  year  doy  hour
 199352        221 2002-01-01 01:00:00  4005.033667  2002    1     1
 199353        221 2002-01-01 01:10:00  4004.793667  2002    1     1
 199354        221 2002-01-01 01:20:00  4004.328667  2002    1     1
 199355        221 2002-01-01 01:30:00  4004.133667  2002    1     1
 199356        221 2002-01-01 01:40:00  4000.343667  2002    1     1
 199357        221 2002-01-01 01:50:00  4000.133667  2002    1     1,
         series_id              

In [32]:
(4013.633667+4009.783667+4005.933667+4005.408667+4005.228667+4005.048667)/6

4007.5061669999996

In [165]:
df = {'index': [], 'series_id': [], 'ts': [], 'year': [], 'doy': [], 'hour': [], 'value': []}

for e in list:
    df['index'].append(e.index[0])
    df['series_id'].append(e.series_id.iloc[0])
    df['ts'].append(e.ts.iloc[0])
    df['year'].append(e.year.iloc[0])
    df['doy'].append(e.doy.iloc[0])
    df['hour'].append(e.hour.iloc[0])
    df['value'].append(e.value.mean())

df_daily = pd.DataFrame(df)
df_daily.set_index('index', inplace=True)
df_daily.index.name = ""

In [166]:
df_daily

,series_id,ts,year,doy,hour,value
,,,,,,
199346,221,2002-01-01 00:00:00,2002,1,0,4007.506167
199352,221,2002-01-01 01:00:00,2002,1,1,4003.127834
199358,221,2002-01-01 02:00:00,2002,1,2,3998.522834
199364,221,2002-01-01 03:00:00,2002,1,3,3993.111167
199370,221,2002-01-01 04:00:00,2002,1,4,3987.445334
...,...,...,...,...,...,...
251876,221,2002-12-31 19:00:00,2002,365,19,6024.210167
251882,221,2002-12-31 20:00:00,2002,365,20,6024.207667
251888,221,2002-12-31 21:00:00,2002,365,21,6024.198167


In [152]:
df_daily.set_index('index', inplace=True)

In [158]:
df_daily.rename(columns = {'index':' '})

,series_id,ts,year,doy,hour,value
index,,,,,,
199346,221,2002-01-01 00:00:00,2002,1,0,4007.506167
199352,221,2002-01-01 01:00:00,2002,1,1,4003.127834
199358,221,2002-01-01 02:00:00,2002,1,2,3998.522834
199364,221,2002-01-01 03:00:00,2002,1,3,3993.111167
199370,221,2002-01-01 04:00:00,2002,1,4,3987.445334
...,...,...,...,...,...,...
251876,221,2002-12-31 19:00:00,2002,365,19,6024.210167
251882,221,2002-12-31 20:00:00,2002,365,20,6024.207667
251888,221,2002-12-31 21:00:00,2002,365,21,6024.198167


In [161]:
df_daily.index.name = ""

In [162]:
df_daily

,series_id,ts,year,doy,hour,value
,,,,,,
199346,221,2002-01-01 00:00:00,2002,1,0,4007.506167
199352,221,2002-01-01 01:00:00,2002,1,1,4003.127834
199358,221,2002-01-01 02:00:00,2002,1,2,3998.522834
199364,221,2002-01-01 03:00:00,2002,1,3,3993.111167
199370,221,2002-01-01 04:00:00,2002,1,4,3987.445334
...,...,...,...,...,...,...
251876,221,2002-12-31 19:00:00,2002,365,19,6024.210167
251882,221,2002-12-31 20:00:00,2002,365,20,6024.207667
251888,221,2002-12-31 21:00:00,2002,365,21,6024.198167


In [ ]:
data = df[1]
data.loc[:, 'year'] = data.ts.dt.year  # note: add a column with the year
data.loc[:, 'doy'] = data.ts.dt.dayofyear  # note: add a column with the day of year
data.loc[:, 'hour'] = data.ts.dt.hour  # note: add a column with the hour of day

new = data.groupby(['year', 'doy', 'hour'])

list = []
indx = []
for a, b in new:
    list.append(b)
    indx.append(a)

names = []
df_temp = {'index': []}
for el in data.columns:
    df_temp[el] = []
    names.append()
    
for e in list:
    df_temp['index'].append(e.index[0])
    df_temp['series_id'].append(e['series_id'].iloc[0])
    df_temp['ts'].append(e['ts'].iloc[0])
    df_temp['year'].append(e['year'].iloc[0])
    df_temp['doy'].append(e['doy'].iloc[0])
    df_temp['hour'].append(e['hour'].iloc[0])
    df_temp['value'].append(e['value'].mean())

df_hourly = pd.DataFrame(df_temp)
df_hourly.set_index('index', inplace=True)
df_hourly.index.name = ""

# NOTE: make sure that the first and last row are whole hours, without minutes (i.e. minutes = 0). 
# The last row might always be a whole hour due to the construction of the function. Check anyway.
# In positive case, remove the row.
if df_hourly.iloc[0].ts.minute != 0:
    indx = df_hourly.iloc[0].name
    df_hourly = df_hourly.drop(index=indx)

if df_hourly.iloc[-1].ts.minute != 0:
    indx = df_hourly.iloc[-1].name
    df_hourly = df_hourly.drop(index=indx)

In [189]:
df_hourly

,series_id,ts,value,year,doy,hour
,,,,,,
2,1,2014-05-21 13:00:00,2242.282262,2014,141,13
8,1,2014-05-21 14:00:00,2233.775940,2014,141,14
14,1,2014-05-21 15:00:00,2228.491836,2014,141,15
20,1,2014-05-21 16:00:00,2230.199130,2014,141,16
26,1,2014-05-21 17:00:00,2238.437131,2014,141,17
...,...,...,...,...,...,...
438905,1,2022-10-04 20:00:00,4966.189183,2022,277,20
438911,1,2022-10-04 21:00:00,4968.803700,2022,277,21
438917,1,2022-10-04 22:00:00,4969.622080,2022,277,22


In [186]:
df_hourly

,series_id,ts,value,year,doy,hour
,,,,,,
2,1,2014-05-21 13:00:00,2245.998500,2014,141,13
8,1,2014-05-21 14:00:00,2237.999367,2014,141,14
14,1,2014-05-21 15:00:00,2228.615700,2014,141,15
20,1,2014-05-21 16:00:00,2229.000000,2014,141,16
26,1,2014-05-21 17:00:00,2234.207200,2014,141,17
...,...,...,...,...,...,...
438905,1,2022-10-04 20:00:00,4965.331478,2022,277,20
438911,1,2022-10-04 21:00:00,4967.984828,2022,277,21
438917,1,2022-10-04 22:00:00,4969.418799,2022,277,22


In [182]:
df_hourly

,series_id,ts,value,year,doy,hour
,,,,,,
2,1,2014-05-21 13:00:00,2242.282262,2014,141,13
8,1,2014-05-21 14:00:00,2233.775940,2014,141,14
14,1,2014-05-21 15:00:00,2228.491836,2014,141,15
20,1,2014-05-21 16:00:00,2230.199130,2014,141,16
26,1,2014-05-21 17:00:00,2238.437131,2014,141,17
...,...,...,...,...,...,...
438905,1,2022-10-04 20:00:00,4966.189183,2022,277,20
438911,1,2022-10-04 21:00:00,4968.803700,2022,277,21
438917,1,2022-10-04 22:00:00,4969.622080,2022,277,22


In [ ]:
is_minute = df_hourly.iloc[0].ts.minute
if df_hourly.iloc[0].ts.minute != 0:
    df_hourly.drop(index = )

True

In [164]:
indx = df_hourly.iloc[0].name

In [165]:
df_hourly = df_hourly.drop(index=indx)

In [4]:
df = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_dendro_lm.pkl")
meta = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_dendrometer.pkl")
meta_all = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")
combined = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/combined_dendro_climate_dictionary.pkl")
climate = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_meteo_l2.pkl")
hourly = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/hourly_dendrometer_dictionary.pkl")

In [30]:
dendro_data = hourly[3]

In [9]:
climate[1]

,site_id,ts,temp,rh,swp,total_precip,rad,vpd,vpd_bo
0,1,2011-10-13 09:00:00,None,None,None,0.000000,None,None,None
1,1,2011-10-13 10:00:00,None,None,None,0.000000,412.166667,None,None
2,1,2011-10-13 11:00:00,None,None,None,0.000000,352.833333,None,None
3,1,2011-10-13 12:00:00,None,None,None,0.000000,452.333333,None,None
4,1,2011-10-13 13:00:00,None,None,None,0.000000,587.166667,None,None
...,...,...,...,...,...,...,...,...,...
96234,1,2022-10-05 03:00:00,8.333333,47.833333,-11.336063,0.000000,1.000000,0.574437,0.575383
96235,1,2022-10-05 04:00:00,8.000000,46.500000,-11.342608,0.000000,0.000000,0.575926,0.576168
96236,1,2022-10-05 05:00:00,8.166667,45.166667,-11.346543,0.000000,0.000000,0.597006,0.598151
96237,1,2022-10-05 06:00:00,8.500000,42.666667,-11.346577,0.000000,7.000000,0.638506,0.638774


In [21]:
meta[["series_id", "site_id"]]

,series_id,site_id
0,1,1
1,2,1
2,3,2
3,4,2
17,18,3
...,...,...
1435,1441,134
1436,1442,134
1446,1452,134
1586,1592,95


In [19]:
combined[3]

,series_id,site_id,ts,stem_radius,temp,rh,swp,total_precip,rad,vpd
0,3,3,2014-05-21 13:00:00,-7.556790,21.000000,35.000000,-10.659580,0.000000,621.104034,1.621833
1,3,3,2014-05-21 14:00:00,-17.293647,21.333333,34.500000,-10.657120,0.000000,441.957845,1.668089
2,3,3,2014-05-21 15:00:00,-21.695306,18.333333,40.500000,-10.657120,0.000000,255.908265,1.258077
3,3,3,2014-05-21 16:00:00,-19.150150,17.666667,42.500000,-10.658370,0.000000,145.055130,1.165872
4,3,3,2014-05-21 17:00:00,-13.129481,17.000000,45.333333,-10.662080,0.000000,74.625712,1.062684
...,...,...,...,...,...,...,...,...,...,...
73436,3,3,2022-10-04 19:00:00,7357.552277,10.666667,55.333333,-11.307812,0.000000,0.000000,0.575394
73437,3,3,2022-10-04 20:00:00,7360.577202,10.166667,55.500000,-11.307812,0.000000,0.000000,0.554434
73438,3,3,2022-10-04 21:00:00,7363.553854,9.666667,54.333333,-11.310425,0.000000,0.000000,0.550224
73439,3,3,2022-10-04 22:00:00,7364.853651,9.166667,53.666667,-11.313021,0.000000,0.000000,0.539792


In [ ]:
climate_data = climate[meta[3].site_id]

In [23]:
meta_id = {}
# NOTE: iterate over the metadata table and convert it to a dictionary in which the series_id is the key.
for index, row in meta.iterrows():    
    meta_id[row.series_id] = row


In [29]:
meta_id[3].site_id

2

In [31]:
clima_data = climate[meta_id[3].site_id]

In [27]:
meta_id[3]

measure_point      alvaneu-low_3_tree-stem-radius-change_1.5_alva...
series_id                                                          3
series_start                                     2014-05-21 00:00:00
series_stop                                      2022-10-05 00:00:00
series_cutout                                                   None
                                         ...                        
site_growth_vpd                                                 0.51
site_n_depo                                                     None
site_ozon                                                       None
site_nfk                                                      191.00
site_temp_ref                                                 1178.0
Name: 2, Length: 106, dtype: object

In [32]:
pd.merge(dendro_data, clima_data, on = "ts", how = "outer")

,series_id,ts,year,doy,hour,value,site_id,temp,rh,swp,total_precip,rad,vpd,vpd_bo
0,NaN,2011-10-13 09:00:00,NaN,NaN,NaN,NaN,2.0,None,None,None,0.000000,None,None,None
1,NaN,2011-10-13 10:00:00,NaN,NaN,NaN,NaN,2.0,None,None,None,0.000000,412.166667,None,None
2,NaN,2011-10-13 11:00:00,NaN,NaN,NaN,NaN,2.0,None,None,None,0.000000,352.833333,None,None
3,NaN,2011-10-13 12:00:00,NaN,NaN,NaN,NaN,2.0,None,None,None,0.000000,452.333333,None,None
4,NaN,2011-10-13 13:00:00,NaN,NaN,NaN,NaN,2.0,None,None,None,0.000000,587.166667,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96273,NaN,2022-10-05 04:00:00,NaN,NaN,NaN,NaN,2.0,8.333333,45.333333,-11.326052,0.000000,0.000000,0.601975,0.602865
96274,NaN,2022-10-05 05:00:00,NaN,NaN,NaN,NaN,2.0,8.166667,43.500000,-11.328660,0.000000,0.000000,0.615162,0.615852
96275,NaN,2022-10-05 06:00:00,NaN,NaN,NaN,NaN,2.0,8.500000,41.833333,-11.329960,0.000000,7.000000,0.647797,0.648058
96276,NaN,2022-10-05 07:00:00,NaN,NaN,NaN,NaN,2.0,8.666667,39.500000,-11.331269,0.000000,24.000000,0.681431,0.681974


In [33]:
dendro_data

,series_id,ts,year,doy,hour,value
,,,,,,
2,3,2014-05-21 13:00:00,2014,141,13,-7.556790
8,3,2014-05-21 14:00:00,2014,141,14,-17.293647
14,3,2014-05-21 15:00:00,2014,141,15,-21.695306
20,3,2014-05-21 16:00:00,2014,141,16,-19.150150
26,3,2014-05-21 17:00:00,2014,141,17,-13.129481
...,...,...,...,...,...,...
431433,3,2022-10-04 20:00:00,2022,277,20,7360.577202
431439,3,2022-10-04 21:00:00,2022,277,21,7363.553854
431445,3,2022-10-04 22:00:00,2022,277,22,7364.853651


In [34]:
clima_data

,site_id,ts,temp,rh,swp,total_precip,rad,vpd,vpd_bo
0,2,2011-10-13 09:00:00,None,None,None,0.000000,None,None,None
1,2,2011-10-13 10:00:00,None,None,None,0.000000,412.166667,None,None
2,2,2011-10-13 11:00:00,None,None,None,0.000000,352.833333,None,None
3,2,2011-10-13 12:00:00,None,None,None,0.000000,452.333333,None,None
4,2,2011-10-13 13:00:00,None,None,None,0.000000,587.166667,None,None
...,...,...,...,...,...,...,...,...,...
96235,2,2022-10-05 04:00:00,8.333333,45.333333,-11.326052,0.000000,0.000000,0.601975,0.602865
96236,2,2022-10-05 05:00:00,8.166667,43.500000,-11.328660,0.000000,0.000000,0.615162,0.615852
96237,2,2022-10-05 06:00:00,8.500000,41.833333,-11.329960,0.000000,7.000000,0.647797,0.648058
96238,2,2022-10-05 07:00:00,8.666667,39.500000,-11.331269,0.000000,24.000000,0.681431,0.681974


In [3]:
import sys
import pandas as pd
import argparse
import pickle
sys.path.insert(0, '../raw_data_elaboration/tools/')
import data_organisation as do

meta = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_dendrometer.pkl")
clima = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_meteo_l2.pkl")
hourly_dendro = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/hourly_dendrometer_dictionary.pkl")

meta_id = {}
# NOTE: iterate over the metadata table and convert it to a dictionary in which the series_id is the key.
for index, row in meta.iterrows():    
    meta_id[row.series_id] = row
df = {}

for key, data in hourly_dendro.items():
        print(key)
        climate_data = clima[meta_id[key].site_id]
        if len(data) > 0 and len(climate_data) > 0:
            a = pd.merge(data, climate_data, on = "ts", how = "outer")  # NOTE: merge the dendrometer data with the climate data
            b = a [a.value.first_valid_index():a.value.last_valid_index()]  # NOTE: remove the head and tail where "value" (in this case, the dendrometer value) is not defined
            if not ( b.temp.isnull().all() and b.vpd.isnull().all() ):  # NOTE: make sure that the merged datasets actually overlap, at least over temperature and vapour pressure difference
                c = b[["series_id", "site_id", "ts", "value", "temp", "rh", "swp", "total_precip", "rad", "vpd"]].rename(columns={"value": "stem_radius"})  # NOTE: select the features to use
                c['site_id'] = c['site_id'].ffill().bfill()  # NOTE: the merging process changes the site and seris ids to double because of the presence of NaNs. This command fills the rows with the corresponding index.
                c['series_id'] = c['series_id'].ffill().bfill()
                
                c['site_id'] = c['site_id'].astype(int) # NOTE: makes sure that the values are integers
                c['series_id'] = c['series_id'].astype(int)

                start = c.ts.iloc[0]
                end = c.ts.iloc[-1]
                complete_times = {'ts': pd.date_range(start=start, end=end, freq='1h')}   # NOTE: alternative: "10Min". For more information see https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#timeseries-offset-aliases
                d = pd.merge(c, pd.DataFrame(complete_times), on = "ts", how = "outer")  # NOTE: make sure that all the time stamps are present
                df[key] = d

with open("combined_dendro_climate_dictionary.pkl", 'wb') as f:
    pickle.dump(df, f)

with open("metadata_dictionary.pkl", 'wb') as f:
    pickle.dump(meta_id, f)

1
2
3
4
18
19
20
21
22
23
24
27
28
29
30
31
32
53
54
72
73
74
75
83
84
120
121
136
137
138
139
156
157
160
161
162
163
166
167
168
169
172
173
192
193
194
195
200
201
202
203
204
205
206
207
208
221
222
223
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
296
320
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
377
378
379
380
405
406
407
408
409
410
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
451
452
453
454
455
456
457
458
459
460
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
490
491
492
493
494
495
496
497
498
499
500
501
502
503
504
505
506
507
508
509
510
511
512
513
514
515
516
517
518
519
520
521
522
523
524
525
526
527
528
529
530
537
538
539
540
541
542
543
544
545
546
557
558
572
573
574
575
576
577
586
587
588
589
590
591
592
593
594
595
661
662
663
664
665
666
667
668
669
670
671
6

In [3]:
meta_id[465].site_id

45

In [8]:
c.iloc[10000:10100]

,series_id,site_id,ts,stem_radius,temp,rh,swp,total_precip,rad,vpd


In [16]:
if b.temp.isnull().all() and b.vpd.isnull().all():
    print(1)
else:
    print(0)

1


In [10]:
clima[45]

,site_id,ts,temp,rh,swp,total_precip,rad,vpd,vpd_bo
0,45,2020-05-28 20:00:00,None,None,None,0.000000,None,None,None
1,45,2020-05-28 21:00:00,None,None,None,0.000000,None,None,None
2,45,2020-05-28 22:00:00,None,None,None,0.000000,None,None,None
3,45,2020-05-28 23:00:00,None,None,None,0.000000,None,None,None
4,45,2020-05-29 00:00:00,None,None,None,0.000000,None,None,None
...,...,...,...,...,...,...,...,...,...
25110,45,2023-04-11 11:00:00,9.698677,76.335538,-5.400000,None,None,0.285830,None
25111,45,2023-04-11 12:00:00,9.854671,73.996731,-5.400000,None,None,0.317383,None
25112,45,2023-04-11 13:00:00,10.202047,68.949241,-5.400000,None,None,0.387907,None
25113,45,2023-04-11 14:00:00,10.998502,65.431679,-5.400000,None,None,0.455390,None


In [5]:
a = 1991
b = 1995
for e in range(a,b):
    print(e)

1991
1992
1993
1994


In [1]:
import pandas as pd
data_all_l1 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_all_l1_dictionary.pkl")

In [3]:
data_all_l1[1]

,series_id,ts,value
0,1,2014-05-21 11:40:00,2249.3757
1,1,2014-05-21 11:50:00,2248.3751
2,1,2014-05-21 12:00:00,2245.9985
3,1,2014-05-21 12:10:00,2243.3328
4,1,2014-05-21 12:20:00,2242.0457
...,...,...,...
438593,1,2022-10-04 23:20:00,4823.6083984375
438594,1,2022-10-04 23:30:00,4823.6083984375
438595,1,2022-10-04 23:40:00,4823.6083984375
438596,1,2022-10-04 23:50:00,4823.6083984375


In [4]:
meta = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")

In [5]:
meta

,measure_point,series_id,series_start,series_stop,series_cutout,series_active,series_display,variable_id,variable_name,variable_resolution,...,site_annual_rad,site_growth_rad,site_annual_relh,site_growth_relh,site_annual_vpd,site_growth_vpd,site_n_depo,site_ozon,site_nfk,site_temp_ref
0,alvaneu-high_1_tree-stem-radius-change_1.5_alv...,1,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,166.52,243.83,70.81,70.57,0.30,0.48,None,None,106.00,1178.0
1,alvaneu-high_2_tree-stem-radius-change_1.5_alv...,2,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,166.52,243.83,70.81,70.57,0.30,0.48,None,None,106.00,1178.0
2,alvaneu-low_3_tree-stem-radius-change_1.5_alva...,3,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,164.93,239.27,70.23,69.62,0.33,0.51,None,None,191.00,1178.0
3,alvaneu-low_4_tree-stem-radius-change_1.5_alva...,4,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,164.93,239.27,70.23,69.62,0.33,0.51,None,None,191.00,1178.0
4,bachtel-forest_5_precipitation_2_bachtel-0.pre...,5,2012-08-24,2022-09-28,None,no,no,10,precipitation,10,...,137.28,223.03,77.13,74.20,0.30,0.50,None,None,None,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1905,bern-drakau_1929_soil-water-potential_-0.4_227...,1929,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1906,bern-drakau_1930_soil-water-potential_-1_22747...,1930,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1907,bern-drakau_1931_soil-water-potential_-0.1_227...,1931,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1908,bern-drakau_1932_soil-water-potential_-0.4_227...,1932,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN


In [ ]:
temperature = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")
humidity = dp..read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")